# Tuba v4 — Welcome & Setup

**Tuba v4** is an AI-ready piping stress analysis & routing library built for engineers and LLM agents alike.  
Every model is described by a **canonical JSON schema** — a single source of truth that can be created by hand, generated by code, or produced by an AI agent.

Under the hood, Tuba provides:

- A **cursor-based DSL** (`PipingBuilder`) for constructing piping geometry in a few lines of Python.
- A **Code_Aster solver backend** for production-grade finite-element stress analysis.
- **ASME B31.3 compliance** evaluation with stress intensification factors.
- **3D visualization** via PyVista, plus HTML/glTF/Blender export.
- **Autorouting** with grid-based pathfinding and obstacle avoidance.

### What you'll learn in this notebook

1. How to install Tuba v4.
2. How to create your first piping model — an L-shaped pipe with two anchors.
3. How to serialize the model to JSON.
4. How to render the pipe in interactive 3D, right here in Jupyter.

## Installation

Tuba v4 requires **Python 3.10+**.

**Core install** (from the repository root):

```bash
pip install -e .
```

**With notebook visualization** (PyVista + Jupyter integration):

```bash
pip install -e ".[notebook-viz]"
```

The `notebook-viz` extra pulls in `pyvista`, `trame`, and everything needed to render interactive 3D views directly inside Jupyter cells.

## Imports & Environment Setup

In [ ]:
import sys
from pathlib import Path

# Ensure the repo root is on sys.path so `import tuba` works
# regardless of where the kernel was started.
REPO_ROOT = Path.cwd()
if REPO_ROOT.name.lower() == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tuba import Model

import pyvista as pv
# Defaults to zoomable 'client' locally; set TUBA_NOTEBOOK_BACKEND=static for nbconvert/CI.
from tuba.visualizer.notebook import configure_notebook_backend
JUPYTER_BACKEND = configure_notebook_backend()

print(f"Tuba v4 loaded from: {REPO_ROOT}")
print(f"PyVista version:     {pv.__version__}")

## Architecture Overview

At the centre of Tuba v4 sits the **`TubaModel`** (aliased as `Model`).  
It is the *single source of truth* for every piping system — geometry, properties, loads, and boundary conditions are all stored in one object.

```
┌─────────────────────────────────────────────┐
│                  TubaModel                  │
├─────────────────────────────────────────────┤
│  Materials      dict[str, Material]         │
│  Sections       dict[str, PipeSection|...]  │
│  Nodes          dict[str, Node]             │
│  Elements       list[Element]               │
│  Supports       dict[str, Support]          │
│  Load Cases     dict[str, LoadCase]         │
│  Tees           dict[str, TeeDefinition]    │
│  Obstacles      list[Obstacle]              │
└─────────────────────────────────────────────┘
```

The entire model can be round-tripped through JSON:

| Method | Description |
|---|---|
| `model.to_dict()` | Convert to a plain Python dictionary |
| `model.to_json(path)` | Write JSON file to disk |
| `Model.from_dict(d)` | Reconstruct from dictionary |
| `Model.from_json(path)` | Load from JSON file |

This makes every Tuba model fully **serialisable, diffable, and AI-parseable**.

## Your First Model — An L-Shaped Pipe

We'll build a simple L-shaped pipe using the `PipingBuilder` DSL:

1. Define a **material** (carbon steel).
2. Define a **pipe section** (DN100 / 4" Sch 40).
3. Use `model.pipe()` to create a builder, then call `start → run → bend → run → end`.

In [ ]:
# --- 1. Create the model container ---
model = Model("HelloPipe")

# --- 2. Material: generic carbon steel ---
model.add_material(
    "CarbonSteel",
    E=2.1e11,       # Young's modulus [Pa]
    nu=0.3,         # Poisson's ratio
    rho=7850,       # Density [kg/m³]
    alpha=1.2e-5,   # Thermal expansion [1/K]
)

# --- 3. Pipe section: DN100 (4" Sch 40) ---
model.add_pipe_section(
    "DN100",
    OD=0.1143,      # Outer diameter [m]
    WT=0.00602,     # Wall thickness [m]
)

# --- 4. Build the L-shaped pipe ---
with model.pipe(section="DN100", material="CarbonSteel") as b:
    b.start([0, 0, 0], support="anchor")   # Fixed anchor at origin
    b.run(3.0)                              # 3 m straight run along +X
    b.bend(radius=0.2, angle=90, plane="XY")  # 90° elbow turning into +Y
    b.run(2.0)                              # 2 m straight run along +Y
    b.end(support="anchor")                 # Fixed anchor at the end

# --- 5. Quick summary ---
print(f"Project:   {model.project_name}")
print(f"Nodes:     {len(model.nodes)}")
print(f"Elements:  {len(model.elements)}")
print(f"Supports:  {len(model.supports)}")

## Inspecting the Model — JSON Serialization

Because Tuba's canonical format is JSON, you can always dump the model to inspect exactly what was built.  
This is the same representation an AI agent would read or produce.

In [ ]:
import json

model_dict = model.to_dict()

# Pretty-print the full model JSON
print(json.dumps(model_dict, indent=2))

## Your First 3D Visualization

Tuba's visualization pipeline converts the model geometry into a **PyVista mesh** that renders interactively inside the notebook.

The two key functions:

| Function | What it does |
|---|---|
| `build_mesh_from_model(model)` | Creates a 1D line mesh with curved bends |
| `inflate_tubes(mesh, radius)` | Extrudes the lines into 3D tubes |

In [ ]:
from tuba.visualizer.pipeline import build_mesh_from_model, inflate_tubes

# Build the line mesh from the model
line_mesh = build_mesh_from_model(model)

# Inflate into 3D tubes (radius = OD / 2)
tube_mesh = inflate_tubes(line_mesh, radius=0.1143 / 2)

# --- Render ---
plotter = pv.Plotter()
plotter.set_background("#1a1a2e")
plotter.add_mesh(tube_mesh, color="#5c6b73", smooth_shading=True)
plotter.add_axes()
plotter.camera.azimuth = 30
plotter.camera.elevation = 20
plotter.show(jupyter_backend=JUPYTER_BACKEND)

## Next Steps

You've just gone from zero to a rendered 3D pipe in under a minute.  
The remaining notebooks in this course dive deeper into every aspect of Tuba v4:

| Notebook | Topic |
|---|---|
| **01** | Building Complex Piping — multi-branch systems, tees, reducers |
| **02** | Supports & Boundary Conditions — anchors, guides, springs, rests |
| **03** | Stress Analysis — Code_Aster solver, ASME B31.3 compliance |
| **04** | Visualization Gallery — stress contours, deformed shapes, HTML export |
| **05** | Autorouting — grid-based pathfinding with obstacle avoidance |
| **06** | Structural Frames — beams, bars, cables, rack bays |
| **07** | BIM Exchange — IFC import/export for interoperability |

Happy piping! 🔧